# Crafter Tabula Rasa JEPA World Model: Google Colab Training Pipeline

This notebook implements the complete **Joint Embedding Predictive Architecture (JEPA)** training pipeline for Crafter.
- **Zero Cheat Policy & Planning**: Operates solely on raw visual RGB pixel observations (64x64x3) without accessing internal game state (`inventory`, `player_pos`, `semantic`, etc.).
- **Anti-Collapse Self-Supervision**: Employs **VICReg** (Invariance, Variance Hinge, Covariance Decorrelation) ensuring non-collapse without pixel decoders.
- **Intrinsic Epistemic Curiosity**: 4-head predictor ensemble disagreement drives exploration and discovery of causal affordances.
- **Two-Tier Persistent Memory**: 70% rolling FIFO + 30% Curated Milestone Archive.
- **Preemption-Resilient Checkpointing**: Saves checkpoints to Google Drive (`/content/drive/MyDrive/crafter_jepa/checkpoints`).

### Cell 1: Environment Verification & Hardware Diagnostics

In [ ]:
# Cell 1: Environment Verification & Hardware Diagnostics
import os
import sys
import platform
import torch

print("=" * 70)
print("  SYSTEM & HARDWARE DIAGNOSTICS")
print("=" * 70)
print(f"Python Version:   {platform.python_version()} ({sys.executable})")
print(f"OS Platform:      {platform.system()} {platform.release()}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version:     {torch.version.cuda}")
    print(f"Device Count:     {torch.cuda.device_count()}")
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    print(f"Total VRAM:       {vram_bytes / (1024**3):.2f} GB")
else:
    print("Running in CPU Mode (Hardware accelerator not detected).")
print("=" * 70)

### Cell 2: Google Drive Mounting & Persistent Checkpoint Storage

In [ ]:
# Cell 2: Google Drive Mounting & Namespace Isolation (Rule 1: Dynamic Auto-Creation)
import os
from pathlib import Path

try:
    from google.colab import drive
    print("[Colab Environment Detected] Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("[Non-Colab / Local Environment] Running locally.")

# Target: /content/drive/MyDrive/learn-a-world/prometheus/checkpoints (or /content/drive/MyDrive/crafter_jepa/checkpoints)
if os.path.exists("/content/drive/MyDrive"):
    BASE_DRIVE = Path("/content/drive/MyDrive/learn-a-world")
    MODEL_SUBDIR = BASE_DRIVE / "prometheus"
    CHECKPOINT_DIR = MODEL_SUBDIR / "checkpoints"
    LOGS_DIR = MODEL_SUBDIR / "logs"
    SHARDS_DIR = MODEL_SUBDIR / "shards"
else:
    CHECKPOINT_DIR = Path("./checkpoints")
    LOGS_DIR = Path("./logs")
    SHARDS_DIR = Path("./shards")

# Rule 1: Dynamic Auto-Creation
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SHARDS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[+] Model checkpoint directory ready: {CHECKPOINT_DIR.resolve()}")
print(f"[+] Logs directory ready: {LOGS_DIR.resolve()}")
print(f"[+] Shards directory ready: {SHARDS_DIR.resolve()}")

### Cell 3: Pip Install Dependencies & Environment Alignment
Installs crafter, gymnasium, and modern ABI-aligned packages.

In [ ]:
# Cell 3: Install Dependencies & Setup Environment
import sys
import subprocess

print("Installing aligned dependencies (NumPy 2.x, SciPy, Crafter, Gymnasium)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "numpy", "scipy", "crafter", "gymnasium>=0.28.0", "matplotlib>=3.5.0"], check=False)
print("[+] Dependencies successfully installed.")

# ABI compatibility shim for NumPy 2.x and SciPy BLAS checks
import numpy as np
try:
    if hasattr(np, "_core") and hasattr(np._core, "_multiarray_umath"):
        if not hasattr(np._core._multiarray_umath, "_blas_supports_fpe"):
            setattr(np._core._multiarray_umath, "_blas_supports_fpe", lambda *args, **kwargs: False)
except Exception:
    pass

import torch
print(f"[+] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Active GPU:    {torch.cuda.get_device_name(0)}")


### Cell 4: Repository Setup & Path Configuration
Clones or pulls the latest Prometheus repository and verifies module imports.

In [ ]:
# Cell 4: Repository Setup & Path Configuration
import os
import sys
import subprocess
from pathlib import Path

# ABI & compatibility shim for NumPy 2.x and SciPy internal array_api_compat
import numpy as np
try:
    if hasattr(np, "_core") and hasattr(np._core, "_multiarray_umath"):
        if not hasattr(np._core._multiarray_umath, "_blas_supports_fpe"):
            setattr(np._core._multiarray_umath, "_blas_supports_fpe", lambda *args, **kwargs: False)
except Exception:
    pass

# Check Gymnasium import ABI compatibility early with helpful prompt
try:
    import gymnasium
except ValueError as e:
    if "numpy.dtype size changed" in str(e):
        print("\n" + "=" * 70)
        print("⚠️ [ACTION REQUIRED] Colab session needs a quick restart after pip updates.")
        print("   Please click menu: Runtime -> Restart session (or press Ctrl+M .)")
        print("   Then re-run Cell 4!")
        print("=" * 70)
        raise RuntimeError("Kernel restart required to apply updated C-ABI. Please click Runtime -> Restart session.") from e

# In Google Colab, ensure repository is cloned and up to date
if os.path.exists("/content"):
    repo_dir = Path("/content/learn-a-world")
    if not (repo_dir / "prometheus" / "src").exists():
        print("[+] Cloning repository from GitHub...")
        subprocess.run(["git", "clone", "https://github.com/sahiltolani30/learn-a-world.git", "/content/learn-a-world"], check=False)
    else:
        print("[+] Repository present, pulling latest updates...")
        subprocess.run(["git", "-C", "/content/learn-a-world", "checkout", "main"], check=False)
        subprocess.run(["git", "-C", "/content/learn-a-world", "pull", "origin", "main"], check=False)

# Locate and configure project root path
cwd = Path(os.getcwd())
candidates = [
    Path("/content/learn-a-world/prometheus"),
    Path("/content/repo/prometheus"),
    Path("/content/prometheus"),
    cwd / "prometheus",
    cwd,
    cwd.parent,
]

REPO_ROOT = cwd
for candidate in candidates:
    if (candidate / "src" / "agent").exists():
        REPO_ROOT = candidate
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if "src" in sys.modules:
    del sys.modules["src"]
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("src."):
        del sys.modules[mod_name]

print(f"[+] Repository root configured: {REPO_ROOT.resolve()}")

# Verify core module imports
try:
    from src.env.wrapper import make_crafter_env
    from src.agent.agent import JEPAAgent
    from src.models.losses import VICRegLoss
    from src.memory.buffer import TwoTierReplayBuffer
    from src.memory.storage import CheckpointManager
    print("[+] Core JEPA modules imported successfully!")
except ImportError as err:
    print(f"[!] Import error: {err}")
    raise


### Cell 5: Fast CPU Smoke Test Verification (<60s)

In [ ]:
# Cell 5: Fast CPU Smoke Test Verification (<60s)
# Validates complete 13-stage JEPA loop (zero-cheat, VICReg, buffer, curiosity)
from src.test_smoke import run_smoke_test

print("Executing fast local CPU smoke test...")
smoke_results = run_smoke_test()
print(f"\n[+] CPU Smoke Test Passed! Elapsed: {smoke_results['total_elapsed_seconds']:.3f} s")

### Cell 6: Full GPU Training Run with Drive Checkpointing

In [ ]:
# Cell 6: Full GPU Training Run with Drive Checkpointing
!git -C /content/learn-a-world pull origin main
import torch
from src.train import TrainingPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Starting Tabula Rasa JEPA Training on device: {device}")

training_config = {
    "device": device,
    "total_steps": 25_000,
    "batch_size": 64,
    "checkpoint_dir": CHECKPOINT_DIR,
    "eval_interval": 1_000,
    "eval_episodes": 3,
    "seed": 42,
    "lr": 1e-4,
    "critic_lr": 3e-4,
    "warmup_steps": 100,
    "save_interval": 1_000,
    "no_resume": False,
    "tier2_ratio": 0.30,
    "gamma": 0.99,
    "tau": 0.99,
    "log_interval": 100,
}

pipeline = TrainingPipeline(training_config)
train_results = pipeline.train()

print("\n[+] Training session completed successfully.")
print(f"Total Steps:        {train_results['total_steps']}")
print(f"Total Episodes:     {train_results['total_episodes']}")
print(f"Achievements Count: {len(train_results['achievements_unlocked'])}")
print(f"Achievements List:  {train_results['achievements_unlocked']}")
print(f"Final Checkpoint:   {train_results['final_checkpoint']}")

# Cell 7: Evaluation Progression & Checkpoint Auto-Loader
import os
import torch
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# 1. Auto-connect to pipeline from Google Drive if kernel was restarted
if "pipeline" not in globals() or pipeline is None:
    print("[*] Reconnecting to saved checkpoint on Google Drive...")
    from src.train import TrainingPipeline
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    ckpt_dir = Path("/content/drive/MyDrive/learn-a-world/prometheus/checkpoints")
    if not ckpt_dir.exists():
        for cand in Path("/content/drive").glob("**/prometheus/checkpoints"):
            ckpt_dir = cand
            break
            
    training_config = {
        "device": device,
        "checkpoint_dir": ckpt_dir,
        "no_resume": False,
    }
    pipeline = TrainingPipeline(training_config)
    print(f"[+] Restored pipeline at step {pipeline.current_step} (Episode {pipeline.current_episode})!")

# 2. Extract and Plot Achievement & Evaluation History
ach_hist = getattr(pipeline, "achievement_history", {})
steps = sorted(list(ach_hist.keys()))

if steps:
    returns = [ach_hist[s]["mean_return"] for s in steps]
    lengths = [ach_hist[s]["mean_length"] for s in steps]
    ach_counts = [ach_hist[s]["achievement_count"] for s in steps]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Project Prometheus Evaluation History ({len(steps)} Milestones, Up to Step {pipeline.current_step})", fontsize=14, fontweight="bold")

    # 1. Evaluation Returns
    ax1 = axes[0]
    ax1.plot(steps, returns, marker="o", color="#27ae60", linewidth=2)
    ax1.axhline(0, color="gray", linestyle="--", alpha=0.6)
    ax1.set_title("Mean Episodic Return", fontsize=11, fontweight="bold")
    ax1.set_xlabel("Environment Steps")
    ax1.set_ylabel("Evaluation Return")
    ax1.grid(True, linestyle="--", alpha=0.5)

    # 2. Survival Length
    ax2 = axes[1]
    ax2.plot(steps, lengths, marker="s", color="#2980b9", linewidth=2)
    ax2.set_title("Mean Survival Length (Steps / Episode)", fontsize=11, fontweight="bold")
    ax2.set_xlabel("Environment Steps")
    ax2.set_ylabel("Survival Steps")
    ax2.grid(True, linestyle="--", alpha=0.5)

    # 3. Achievements Unlocked in Eval
    ax3 = axes[2]
    ax3.plot(steps, ach_counts, marker="^", color="#8e44ad", linewidth=2)
    ax3.set_title("Achievements Unlocked During Eval", fontsize=11, fontweight="bold")
    ax3.set_xlabel("Environment Steps")
    ax3.set_ylabel("Unique Achievements")
    ax3.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

    print(f"\n[+] Highest Evaluation Return: {max(returns):.2f} (Recorded at Step {steps[np.argmax(returns)]})")
    print(f"[+] Longest Survival Average:  {max(lengths):.1f} steps (Recorded at Step {steps[np.argmax(lengths)]})")
else:
    print(f"[+] Loaded pipeline at step {pipeline.current_step}. Run Cell 8 and Cell 10 for deep diagnostics and benchmark!")


In [ ]:
# Cell 7: Evaluation Progression & Checkpoint Auto-Loader
!git -C /content/learn-a-world pull origin main
import os
import torch
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# 1. Auto-connect to pipeline from Google Drive if kernel was restarted
if "pipeline" not in globals() or pipeline is None:
    print("[*] Reconnecting to saved checkpoint on Google Drive...")
    from src.train import TrainingPipeline
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    ckpt_dir = Path("/content/drive/MyDrive/learn-a-world/prometheus/checkpoints")
    if not ckpt_dir.exists():
        for cand in Path("/content/drive").glob("**/prometheus/checkpoints"):
            ckpt_dir = cand
            break
            
    training_config = {
        "device": device,
        "checkpoint_dir": ckpt_dir,
        "no_resume": False,
    }
    pipeline = TrainingPipeline(training_config)
    print(f"[+] Restored pipeline at step {pipeline.current_step} (Episode {pipeline.current_episode})!")

# 2. Extract and Plot Achievement & Evaluation History
ach_hist = getattr(pipeline, "achievement_history", {})
steps = sorted(list(ach_hist.keys()))

if steps:
    returns = [ach_hist[s]["mean_return"] for s in steps]
    lengths = [ach_hist[s]["mean_length"] for s in steps]
    ach_counts = [ach_hist[s]["achievement_count"] for s in steps]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Project Prometheus Evaluation History ({len(steps)} Milestones, Up to Step {pipeline.current_step})", fontsize=14, fontweight="bold")

    # 1. Evaluation Returns
    ax1 = axes[0]
    ax1.plot(steps, returns, marker="o", color="#27ae60", linewidth=2)
    ax1.axhline(0, color="gray", linestyle="--", alpha=0.6)
    ax1.set_title("Mean Episodic Return", fontsize=11, fontweight="bold")
    ax1.set_xlabel("Environment Steps")
    ax1.set_ylabel("Evaluation Return")
    ax1.grid(True, linestyle="--", alpha=0.5)

    # 2. Survival Length
    ax2 = axes[1]
    ax2.plot(steps, lengths, marker="s", color="#2980b9", linewidth=2)
    ax2.set_title("Mean Survival Length (Steps / Episode)", fontsize=11, fontweight="bold")
    ax2.set_xlabel("Environment Steps")
    ax2.set_ylabel("Survival Steps")
    ax2.grid(True, linestyle="--", alpha=0.5)

    # 3. Achievements Unlocked in Eval
    ax3 = axes[2]
    ax3.plot(steps, ach_counts, marker="^", color="#8e44ad", linewidth=2)
    ax3.set_title("Achievements Unlocked During Eval", fontsize=11, fontweight="bold")
    ax3.set_xlabel("Environment Steps")
    ax3.set_ylabel("Unique Achievements")
    ax3.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()

    print(f"\n[+] Highest Evaluation Return: {max(returns):.2f} (Recorded at Step {steps[np.argmax(returns)]})")
    print(f"[+] Longest Survival Average:  {max(lengths):.1f} steps (Recorded at Step {steps[np.argmax(lengths)]})")
else:
    print(f"[+] Loaded pipeline at step {pipeline.current_step}. Run Cell 8 and Cell 10 for deep diagnostics and benchmark!")


### Cell 8: A-to-Z Forensic Diagnostic & Agent X-Ray Inspection
Inspects visual attention saliency, Q-value spectrum across all 17 actions, world model uncertainty, and Two-Tier replay memory retention.

In [ ]:
# Cell 8: A-to-Z Forensic Diagnostic & Agent X-Ray Inspection
import matplotlib.pyplot as plt
from src.diagnostics import AgentForensics
from src.env.wrapper import make_crafter_env

# 1. Initialize Forensic Inspector with trained agent
forensics = AgentForensics(pipeline.agent)

# 2. Inspect Memory Buffer health & retained milestones
mem_stats = forensics.inspect_memory(pipeline.buffer)
print("=" * 60)
print("  TWO-TIER REPLAY MEMORY FORENSICS")
print("=" * 60)
print(f"Total Transitions in Memory: {mem_stats['total_transitions']}")
print(f"Tier 1 (Rolling FIFO):        {mem_stats['tier1_count']} / {mem_stats['tier1_capacity']}")
print(f"Tier 2 (Curated Milestones):  {mem_stats['tier2_count']} / {mem_stats['tier2_capacity']} ({mem_stats['tier2_utilization_pct']:.1f}%)")
print(f"Retained Milestone Catalog:   {mem_stats['milestone_catalog']}")
print("=" * 60)

# 3. Run a Live Diagnostic Step on a Fresh Crafter Environment
env = make_crafter_env(seed=123)
obs = env.reset()

step_diag = forensics.explain_step(obs, compute_saliency=True)

print("\n" + "=" * 60)
print("  AGENT DECISION SPECTRUM (Top 5 Actions)")
print("=" * 60)
for item in step_diag["ranked_actions"][:5]:
    print(f"Rank {item['rank']}: {item['name']:<18} | Q-Val: {item['q_value']:+.4f} | Prob: {item['prob']:.3f} | Disagreement: {item['uncertainty']:.4f}")
print("=" * 60)
print(f"[+] Agent Chose: '{step_diag['best_action_name']}'")

# 4. Render 4-Panel Visual Forensic Card
fig = forensics.plot_forensic_panel(obs, step_diag, title_suffix="on Step 0")
plt.show()


### Cell 9: Causal Memory & Prerequisite Dependency Inspector
Audits the 6,000+ curated milestone transitions in Tier 2 and tests the Bellman value chain propagation across crafting actions.

In [ ]:
# Cell 9: Causal Memory & Prerequisite Dependency Inspector
import os
import torch
import numpy as np
from pathlib import Path

# 1. Auto-connect to pipeline if not in memory
if "pipeline" not in globals() or pipeline is None:
    print("[*] Reconnecting to saved checkpoint on Google Drive...")
    from src.train import TrainingPipeline
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt_dir = Path("/content/drive/MyDrive/learn-a-world/prometheus/checkpoints")
    if not ckpt_dir.exists():
        for cand in Path("/content/drive").glob("**/prometheus/checkpoints"):
            ckpt_dir = cand
            break
    pipeline = TrainingPipeline({"device": device, "checkpoint_dir": ckpt_dir, "no_resume": False})
    print(f"[+] Loaded pipeline at step {pipeline.current_step}!")

from src.diagnostics import AgentForensics

buffer = pipeline.buffer
agent = pipeline.agent
forensics = AgentForensics(agent)

print("=" * 65)
print("  🧠 TWO-TIER REPLAY MEMORY AUDIT AT STEP 17,000")
print("=" * 65)
mem_stats = forensics.inspect_memory(buffer)
print(f"Total Experiences in Buffer: {mem_stats['total_transitions']}")
print(f"Tier 1 (Rolling FIFO):        {mem_stats['tier1_count']} / {mem_stats['tier1_capacity']}")
print(f"Tier 2 (Curated Milestones):  {mem_stats['tier2_count']} / {mem_stats['tier2_capacity']} ({mem_stats['tier2_utilization_pct']:.1f}%)")
print(f"Tier 2 Milestones Archived:   {mem_stats.get('tier2_milestones', buffer.t2_size)}")
print(f"Novelty Curiosity Threshold:  {mem_stats['novelty_threshold']:.4f}")
print("=" * 65)

# 2. Action Breakdown in Tier 2
t2_size = buffer.t2_size
t2_actions = buffer.t2_actions[:t2_size]
t2_rewards = buffer.t2_rewards[:t2_size]
unique_actions, counts = np.unique(t2_actions, return_counts=True)
action_names = forensics.action_names

print("\n[+] Top Actions Stored in Tier 2 Permanent Memory:")
action_counts = sorted(zip(unique_actions, counts), key=lambda x: -x[1])
for a_idx, cnt in action_counts[:10]:
    name = action_names[a_idx] if a_idx < len(action_names) else f"act_{a_idx}"
    pct = (cnt / t2_size) * 100.0
    print(f"  * {name:<18}: {cnt:>5} transitions ({pct:4.1f}% of Tier 2)")

# 3. Rewarded Discovery Steps in Tier 2
rewarding_indices = np.where(t2_rewards > 0)[0]
print(f"\n[+] Total Rewarded Discovery Transitions in Tier 2: {len(rewarding_indices)}")
if len(rewarding_indices) > 0:
    print(f"    Max Reward Recorded: {np.max(t2_rewards):.2f} | Average Reward: {np.mean(t2_rewards[rewarding_indices]):.2f}")

# 4. Value Chain Propagation across milestone states
with torch.no_grad():
    sample = buffer.sample(batch_size=32, tier2_ratio=1.0, device=agent.device)
    z_sample = agent.encoder(sample["obs"])
    q1, q2 = agent.critic(z_sample)
    q_vals = torch.min(q1, q2).mean(dim=0).cpu().numpy()

print("\n" + "=" * 65)
print("  ⚡ VALUE CHAIN PROPAGATION ACROSS MILESTONE STATES")
print("=" * 65)
crafting_actions = ["do", "place_table", "place_furnace", "make_wood_pickaxe", "make_stone_pickaxe", "make_wood_sword"]
for act in crafting_actions:
    idx = action_names.index(act)
    print(f"  * {act:<18}: Average Q-Value = {q_vals[idx]:+.4f}")
print("=" * 65)
best_craft = max(crafting_actions, key=lambda a: q_vals[action_names.index(a)])
print(f"[+] Highest Valued Crafting Action in Memory: '{best_craft}' (Q = {q_vals[action_names.index(best_craft)]:+.4f})")


### Cell 10: Official Crafter Benchmark Evaluation (20 Independent Games)
Runs 20 full evaluation games, computes the official Crafter Score and per-game averages, and plots the 22-achievement success rate bar chart.

In [ ]:
# Cell 10: Official Crafter Benchmark Evaluation (20 Games)
import os
import torch
from pathlib import Path
import matplotlib.pyplot as plt

# 1. Auto-connect to pipeline if not in memory
if "pipeline" not in globals() or pipeline is None:
    print("[*] Reconnecting to saved checkpoint on Google Drive...")
    from src.train import TrainingPipeline
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt_dir = Path("/content/drive/MyDrive/learn-a-world/prometheus/checkpoints")
    if not ckpt_dir.exists():
        for cand in Path("/content/drive").glob("**/prometheus/checkpoints"):
            ckpt_dir = cand
            break
    pipeline = TrainingPipeline({"device": device, "checkpoint_dir": ckpt_dir, "no_resume": False})
    print(f"[+] Loaded pipeline at step {pipeline.current_step}!")

from src.diagnostics import AgentForensics
forensics = AgentForensics(pipeline.agent)

NUM_GAMES = 20
print(f"Running official evaluation across {NUM_GAMES} independent Crafter games...")
bench = forensics.run_benchmark(num_episodes=NUM_GAMES, planning=False)

print("\n" + "=" * 65)
print(f"  🏆 OFFICIAL CRAFTER BENCHMARK RESULTS ({NUM_GAMES} GAMES)")
print("=" * 65)
print(f"Official Crafter Score:          {bench['crafter_score']:.2f}%")
print(f"Average Achievements Per Game:   {bench['mean_achievements_per_game']:.2f}")
print(f"Average Survival Steps Per Game: {bench['mean_length']:.1f} ± {bench['std_length']:.1f} steps")
print(f"Average Return Per Game:         {bench['mean_return']:.2f} ± {bench['std_return']:.2f}")
print(f"Total Unique Achievements:       {bench['total_unique_achievements']} / 22")
print(f"Unlocked in Eval Games:          {bench['unlocked_list']}")
print("=" * 65)

# Plot Official 22-Achievement Success Rates Bar Chart
fig = forensics.plot_benchmark_results(bench)
plt.show()
